In [4]:
import torch
import torchvision.models as models
import time
import os

# Baseline inference

In [6]:
def benchmark(model, dummy_input=None, num_runs=100):
    if dummy_input is None:
        dummy_input = torch.randn(1, 3, 224, 224)
    
    start = time.time()
    # Warmup
    for _ in range(3):
        with torch.no_grad():
            model(dummy_input)
    end = time.time()
    warmup_time = (end - start)/3

    # Benchmark
    start = time.time()
    for _ in range(num_runs):
        with torch.no_grad():
            model(dummy_input)
    end = time.time()

    avg_time = (end - start) / num_runs
    return warmup_time, avg_time

In [7]:
model_fp32 = models.resnet18(pretrained=True).eval()
dummy_input = torch.randn(1, 3, 224, 224)

# Benchmark the FP32 model
warmup_time, avg_time = benchmark(model_fp32, dummy_input)

print(f"Warmup Time: {warmup_time:.4f} seconds")
print(f"Avg Inference Time: {avg_time:.4f} seconds")

# Benchmark and save
torch.save(model_fp32.state_dict(), "fp32.pth")
print("FP32 size (MB):", os.path.getsize("fp32.pth") / 1e6)

Warmup Time: 0.0233 seconds
Avg Inference Time: 0.0219 seconds
FP32 size (MB): 46.828292


# Pruning

In [5]:
# TODO

# Quantization

## Dynamic Quantization

In [9]:
from torch.quantization import quantize_dynamic

model_dynamic = quantize_dynamic(model_fp32, dtype=torch.qint8)


warmup_time, avg_time = benchmark(model_dynamic, dummy_input)
print(f"Warmup Time: {warmup_time:.4f} seconds")
print(f"Avg Inference Time: {avg_time:.4f} seconds")

torch.save(model_dynamic.state_dict(), "dynamic.pth")
print("Dynamic Quantized size (MB):", os.path.getsize("dynamic.pth") / 1e6)
# Benchmark the dynamic quantized model


Warmup Time: 0.0250 seconds
Avg Inference Time: 0.0214 seconds
Dynamic Quantized size (MB): 45.299578


## Static Quantization

In [20]:
import torch
from torchvision.models import resnet18
from torch.quantization import get_default_qconfig, fuse_modules, prepare, convert

torch.backends.quantized.engine = 'fbgemm'

model = resnet18(pretrained=True).eval()

# Fuse first block (as a test)
fuse_modules(model, [['conv1', 'bn1', 'relu']], inplace=True)
model.qconfig = get_default_qconfig('fbgemm')

# Prepare and convert
model_prepared = prepare(model)
with torch.no_grad():
    model_prepared(torch.randn(1, 3, 224, 224))  # Calibrate

model_quantized = convert(model_prepared)

torch.save(model_quantized.state_dict(), "static.pth")
print("Static Quantized size (MB):", os.path.getsize("static.pth") / 1e6)

# Benchmark will not work on Windows
# benchmark(model_quantized)


C:\Users\Pawel\PycharmProjects\color-analysis\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Pawel\PycharmProjects\color-analysis\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\Pawel\PycharmProjects\color-analysis\.venv\Lib\site-packages\torch\ao\quantization\observer.py:221: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Static Quantized size (MB): 11.920414


## Quantization Aware Training (QAT)
`note: this introduction was heavily inspired by the official PyTorch documentation on quantization`
#### Overview:
Quantization Aware Training (QAT) simulates the effects of quantization during training, allowing a neural network to learn to be robust to quantization noise. Unlike post-training quantization, QAT can maintain almost the same accuracy as full precision models — especially for convolutional neural networks (CNNs) and edge-deployed models.

#### Motivation:
Post-training quantization can cause significant accuracy drops for some models (especially CNNs).
QAT fixes this by training with quantization noise from the start.

#### Mechanism:
    graph TD
    A[Float32 Model] --> B[Insert FakeQuant Modules] 
    B --> C[Train with Quantization Noise]
    C --> D[Convert to INT8 Model]
    D --> E[Deploy on Edge Device]

QAT as pseudocode
```python
# define a floating point model where some layers could benefit from QAT
class M(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # QuantStub converts tensors from floating point to quantized
        self.quant = torch.ao.quantization.QuantStub()
        self.conv = torch.nn.Conv2d(1, 1, 1)
        self.bn = torch.nn.BatchNorm2d(1)
        self.relu = torch.nn.ReLU()
        # DeQuantStub converts tensors from quantized to floating point
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.dequant(x)
        return x

# create a model instance
model_fp32 = M()

# model must be set to eval for fusion to work
model_fp32.eval()

# attach a global qconfig, which contains information about what kind
# of observers to attach. Use 'x86' for server inference and 'qnnpack'
# for mobile inference. Other quantization configurations such as selecting
# symmetric or asymmetric quantization and MinMax or L2Norm calibration techniques
# can be specified here.
# Note: the old 'fbgemm' is still available but 'x86' is the recommended default
# for server inference.
# model_fp32.qconfig = torch.ao.quantization.get_default_qconfig('fbgemm')
model_fp32.qconfig = torch.ao.quantization.get_default_qat_qconfig('x86')

# fuse the activations to preceding layers, where applicable
# this needs to be done manually depending on the model architecture
model_fp32_fused = torch.ao.quantization.fuse_modules(model_fp32,
    [['conv', 'bn', 'relu']])

# Prepare the model for QAT. This inserts observers and fake_quants in
# the model needs to be set to train for QAT logic to work
# the model that will observe weight and activation tensors during calibration.
model_fp32_prepared = torch.ao.quantization.prepare_qat(model_fp32_fused.train())

# run the training loop (not shown)
training_loop(model_fp32_prepared)

# Convert the observed model to a quantized model. This does several things:
# quantizes the weights, computes and stores the scale and bias value to be
# used with each activation tensor, fuses modules where appropriate,
# and replaces key operators with quantized implementations.
model_fp32_prepared.eval()
model_int8 = torch.ao.quantization.convert(model_fp32_prepared)

# run the model, relevant calculations will happen in int8
res = model_int8(input_fp32)
```

## Knowledge(Model) Distillation
`note: this introduction was heavily inspired by the official PyTorch documentation on knowledge distillation`
#### Overview:
Knowledge Distillation is a model compression technique where a smaller model (the student) learns to mimic a larger, more powerful model (the teacher). The student model is trained not just on the ground truth labels, but also on the soft targets (probability distributions) produced by the teacher model.

#### Motivation:
Larger models (e.g., Transformers, ResNets) often have excellent performance, but are too slow or large for deployment.
Knowledge Distillation helps transfer their knowledge to smaller, faster models without substantial accuracy loss.

#### Mechanism:
    graph TD
    A[Input Data] --> B[Teacher Model (Pretrained)]
    A --> C[Student Model (Trainable)]
    B --> D[Soft Targets (High-T Softmax)]
    C --> E[Student Predictions (High-T)]
    D --> F[KL Loss]
    E --> F
    F --> G[Total Loss (with CE)]

model distillation as pseudocode
```python
def distillation_loss(student_logits, teacher_logits, labels, T=2.0, alpha=0.5):
    # Standard cross-entropy loss with ground truth
    ce_loss = F.cross_entropy(student_logits, labels)
    
    # Softened probabilities
    student_probs = F.log_softmax(student_logits / T, dim=1)
    teacher_probs = F.softmax(teacher_logits / T, dim=1)
    
    # KL divergence between softened outputs
    kd_loss = F.kl_div(student_probs, teacher_probs, reduction='batchmean') * (T * T)
    
    return alpha * ce_loss + (1 - alpha) * kd_loss


teacher = ResNet50(pretrained=True).eval()  # Larger, accurate
student = ResNet18()                        # Smaller, faster

for batch in dataloader:
    inputs, labels = batch
    with torch.no_grad():
        teacher_outputs = teacher(inputs)
    student_outputs = student(inputs)
    
    loss = distillation_loss(student_outputs, teacher_outputs, labels)
    loss.backward()
    optimizer.step()
```

## ONNX